In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_clean = spark.table("delitos.silver.delitos_cdmx_silver")
df_clean.count()

In [0]:

# Window functions para features de contexto histórico
w_alcaldia_hora = Window.partitionBy("AlcaldiaHechos", "hora_del_dia")
w_alcaldia_dia = Window.partitionBy("AlcaldiaHechos", "dia_semana")
w_alcaldia_mes = Window.partitionBy("AlcaldiaHechos", "mes_hechos_num")

df_gold = df_clean \
    .withColumn("conteo_alcaldia_hora", F.count("*").over(w_alcaldia_hora)) \
    .withColumn("conteo_alcaldia_dia", F.count("*").over(w_alcaldia_dia)) \
    .withColumn("conteo_alcaldia_mes", F.count("*").over(w_alcaldia_mes))

In [0]:
df_gold = df_gold.select(
    "categoria_delito",
    "hora_del_dia",
    "dia_semana",
    "dia_mes",
    "trimestre",
    "mes_hechos_num",
    "AlcaldiaHechos",
    "longitud",
    "latitud",
    "dias_para_registro",
    "conteo_alcaldia_hora",
    "conteo_alcaldia_dia",
    "conteo_alcaldia_mes"
)

In [0]:
df_gold.show(5, truncate=False)
df_gold.count()

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("delitos.gold.delitos_cdmx_features")

In [0]:
spark.table("delitos.gold.delitos_cdmx_features").count()